# Import libraries

In [15]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier

# Data pre-processing

## Listings

In [16]:
listings = pd.read_csv('data/listings.csv')
#listings.info(show_counts=True)

In [17]:
# Remove completely inactive listings
listings = listings[listings['availability_365'] > 0]


In [18]:
# Filter listing features
listings = listings[['id','neighbourhood_cleansed', 'price', 'latitude', 'longitude']]

# Drop NAs in price
listings.dropna(subset=['price'], inplace=True)

# Update price from string to float64
listings['price'] = listings['price'].str.replace('$', '').str.replace(',', '')
listings['price'] = listings['price'].astype('float64')

# Rename column
listings.rename(columns={'neighbourhood_cleansed':'neighborhood'}, inplace=True)

listings.info()
listings.head(3)



<class 'pandas.DataFrame'>
Index: 6937 entries, 0 to 7768
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            6937 non-null   int64  
 1   neighborhood  6937 non-null   str    
 2   price         6937 non-null   float64
 3   latitude      6937 non-null   float64
 4   longitude     6937 non-null   float64
dtypes: float64(3), int64(1), str(1)
memory usage: 325.2 KB


,id,neighborhood,price,latitude,longitude
0,1188641052152277018,University District,122.32,47.663421,-122.314419
2,599205596133828323,Minor,68.30,47.611710,-122.313350
3,846763,Ravenna,66.58,47.679950,-122.308590


## Calendar

In [19]:
# Note: Calendar is forward-looking. Scraped date: 6/15/2026
calendar = pd.read_csv('data/calendar.csv')

# Select needed features
calendar = calendar[['listing_id', 'date', 'available']]

# Encode target variable
# 0 = Vacant (Low Demand), 1 = Booked/Taken (High Demand)
calendar['is_booked'] = calendar['available'].map({'t': 0, 'f': 1})

# Drop "available" column
calendar = calendar.drop(columns='available')

# Convert date to datetime
calendar['date'] = pd.to_datetime(calendar['date'])

# Slice a 30 day window to ensure data integrity (sparse/inaccurate data if listing is too far out)
calendar_30 = calendar[(calendar['date'] >= '2026-08-01') & (calendar['date'] <= '2026-08-31')].copy()

# Add column for day of week
calendar_30['day_of_week'] = calendar_30['date'].dt.day_name()

calendar_30.info(show_counts=True)
calendar_30.head(3)


<class 'pandas.DataFrame'>
Index: 242234 entries, 46 to 2851821
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   listing_id   242234 non-null  int64         
 1   date         242234 non-null  datetime64[us]
 2   is_booked    242234 non-null  int64         
 3   day_of_week  242234 non-null  str           
dtypes: datetime64[us](1), int64(2), str(1)
memory usage: 9.2 MB


,listing_id,date,is_booked,day_of_week
46,6606,2026-08-01,0,Saturday
47,6606,2026-08-02,0,Sunday
48,6606,2026-08-03,0,Monday


## Join #1 (calendar_30 and listings)

In [20]:
# Join calendar_30 and listings

df = pd.merge(calendar_30, listings, left_on='listing_id', right_on='id', how='inner')

df.info()
df.head(3)


<class 'pandas.DataFrame'>
RangeIndex: 215047 entries, 0 to 215046
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   listing_id    215047 non-null  int64         
 1   date          215047 non-null  datetime64[us]
 2   is_booked     215047 non-null  int64         
 3   day_of_week   215047 non-null  str           
 4   id            215047 non-null  int64         
 5   neighborhood  215047 non-null  str           
 6   price         215047 non-null  float64       
 7   latitude      215047 non-null  float64       
 8   longitude     215047 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(3), str(2)
memory usage: 14.8 MB


,listing_id,date,is_booked,day_of_week,id,neighborhood,price,latitude,longitude
0,6606,2026-08-01,0,Saturday,6606,Wallingford,111.77,47.65444,-122.33629
1,6606,2026-08-02,0,Sunday,6606,Wallingford,111.77,47.65444,-122.33629
2,6606,2026-08-03,0,Monday,6606,Wallingford,111.77,47.65444,-122.33629


## Ticketmaster json file (Seattle 8/1 - 8/31/2026)

In [21]:
# Load the file
with open('data/response.json', 'r') as file:
    data = json.load(file)

events_list = data.get('_embedded', {}).get('events', [])

# Initialize target columns
names = []
dates = []
venue_names = []
lats = []
longs = []

for event in events_list:
    # Event Core Data
    names.append(event.get('name', None))
    dates.append(event.get('dates', {}).get('start', {}).get('localDate', None))
        
    # Venue Details
    venues = event.get('_embedded', {}).get('venues', [])
    if isinstance(venues, list) and len(venues) > 0:
        v = venues[0]  # Grab the primary venue object cleanly
        venue_names.append(v.get('name', None))
        lats.append(v.get('location', {}).get('latitude', None))
        longs.append(v.get('location', {}).get('longitude', None))
    else:
        venue_names.append(None)
        lats.append(None)
        longs.append(None)

# Build the final structured DataFrame
tm_df = pd.DataFrame({
    'event_name': names,
    'event_date': dates,
    'venue_name': venue_names,
    'venue_latitude': lats,
    'venue_longitude': longs
})

# Convert coordinates into numeric float data types safely
tm_df['venue_latitude'] = pd.to_numeric(tm_df['venue_latitude'], errors='coerce')
tm_df['venue_longitude'] = pd.to_numeric(tm_df['venue_longitude'], errors='coerce')


In [22]:
# Convert time to datetime
tm_df['event_date'] = pd.to_datetime(tm_df['event_date'], errors='coerce')

tm_df = tm_df.sort_values(by='event_date')

tm_df

,event_name,event_date,venue_name,venue_latitude,venue_longitude
3,Ed Sheeran: LOOP Tour,2026-08-01,Lumen Field,47.595083,-122.331607
2,Megan Moroney: The Cloud 9 Tour,2026-08-02,Climate Pledge Arena,47.622126,-122.354016
4,"Lionel Richie and Earth, Wind & Fire - Sing a ...",2026-08-03,Climate Pledge Arena,47.622126,-122.354016
5,Benson Boone: Wanted Man Tour,2026-08-04,Climate Pledge Arena,47.622126,-122.354016
6,Meghan Trainor: The Get In Girl Tour,2026-08-05,Climate Pledge Arena,47.622126,-122.354016
13,IVE WORLD TOUR <SHOW WHAT I AM> IN SEATTLE,2026-08-07,Climate Pledge Arena,47.622126,-122.354016
1,Seahawks Football Fest,2026-08-08,Lumen Field,47.595083,-122.331607
10,Seattle Storm vs. Chicago Sky,2026-08-10,Climate Pledge Arena,47.622126,-122.354016
12,YOASOBI: Never Ending Stories Tour,2026-08-12,Climate Pledge Arena,47.622126,-122.354016
14,Seattle Storm vs. Portland Fire,2026-08-14,Climate Pledge Arena,47.622126,-122.354016


In [23]:
# Join venue capacity (sourced from Wikipedia)

capacities = pd.DataFrame([
    {'venue_name': 'Climate Pledge Arena', 'max_capacity': 17151},
    {'venue_name': 'Lumen Field', 'max_capacity': 68740},
    {'venue_name': 'T-Mobile Park', 'max_capacity': 47368}
])

tm_df = pd.merge(tm_df, capacities, how='left', on='venue_name')

tm_df.tail()

,event_name,event_date,venue_name,venue_latitude,venue_longitude,max_capacity
15,Seattle Storm vs. Toronto Tempo,2026-08-26,Climate Pledge Arena,47.622126,-122.354016,17151
16,The Red Clay Strays,2026-08-27,Climate Pledge Arena,47.622126,-122.354016,17151
17,The Strokes - Reality Awaits North America,2026-08-28,Climate Pledge Arena,47.622126,-122.354016,17151
18,Seattle Storm vs. Los Angeles Sparks,2026-08-30,Climate Pledge Arena,47.622126,-122.354016,17151
19,Noah Kahan: The Great Divide Tour,2026-08-30,T-Mobile Park,47.590333,-122.332850,47368


In [24]:
# Upstream aggregation group
def upstream_agg(group):
    return pd.Series({
        'global_event_count': len(group),
        'total_daily_capacity': group['max_capacity'].sum(),
        # Pack the coordinate pairs and capacities into a list of tuples for step 2
        'geo_payload': list(zip(group['venue_latitude'], group['venue_longitude'], group['max_capacity']))
    })

df_events_aggregated = tm_df.groupby('event_date').apply(upstream_agg).reset_index()

df_events_aggregated.tail()

,event_date,global_event_count,total_daily_capacity,geo_payload
13,2026-08-25,1,17151,"[(47.6221261, -122.35401604, 17151)]"
14,2026-08-26,2,85891,"[(47.595083, -122.331607, 68740), (47.6221261,..."
15,2026-08-27,1,17151,"[(47.6221261, -122.35401604, 17151)]"
16,2026-08-28,1,17151,"[(47.6221261, -122.35401604, 17151)]"
17,2026-08-30,2,64519,"[(47.6221261, -122.35401604, 17151), (47.59033..."


## Join #2 (ticketmaster features)

In [25]:
# Join with Ticketmaster df on date
final_df = pd.merge(df, df_events_aggregated, how='left', left_on='date', right_on='event_date')
final_df.info()
final_df.head(3)

<class 'pandas.DataFrame'>
RangeIndex: 215047 entries, 0 to 215046
Data columns (total 13 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   listing_id            215047 non-null  int64         
 1   date                  215047 non-null  datetime64[us]
 2   is_booked             215047 non-null  int64         
 3   day_of_week           215047 non-null  str           
 4   id                    215047 non-null  int64         
 5   neighborhood          215047 non-null  str           
 6   price                 215047 non-null  float64       
 7   latitude              215047 non-null  float64       
 8   longitude             215047 non-null  float64       
 9   event_date            124866 non-null  datetime64[us]
 10  global_event_count    124866 non-null  float64       
 11  total_daily_capacity  124866 non-null  float64       
 12  geo_payload           124866 non-null  object        
dtypes: datetim

,listing_id,date,is_booked,day_of_week,id,neighborhood,price,latitude,longitude,event_date,global_event_count,total_daily_capacity,geo_payload
0,6606,2026-08-01,0,Saturday,6606,Wallingford,111.77,47.65444,-122.33629,2026-08-01,1.0,68740.0,"[(47.595083, -122.331607, 68740)]"
1,6606,2026-08-02,0,Sunday,6606,Wallingford,111.77,47.65444,-122.33629,2026-08-02,1.0,17151.0,"[(47.6221261, -122.35401604, 17151)]"
2,6606,2026-08-03,0,Monday,6606,Wallingford,111.77,47.65444,-122.33629,2026-08-03,1.0,17151.0,"[(47.6221261, -122.35401604, 17151)]"


In [26]:
import numpy as np
from haversine import haversine, Unit

def compute_weighted_distance(row):
    payload = row['geo_payload']
    
    # Safe check for valid payload
    if not isinstance(payload, list) or len(payload) == 0:
        return np.nan
        
    # Origin point as a (lat, lon) tuple in degrees
    origin = (row['latitude'], row['longitude'])
    
    distances = []
    capacities = []
    
    # Calculate distances using the haversine package
    for ev_lat, ev_lon, capacity in payload:
        destination = (ev_lat, ev_lon)
        
        # Computes distance in miles directly
        dist = haversine(origin, destination, unit=Unit.MILES)
        
        distances.append(dist)
        capacities.append(capacity)
        
    # Convert to numpy arrays for calculation
    distances = np.array(distances)
    capacities = np.array(capacities)
    sum_capacity = np.sum(capacities)
    
    if sum_capacity > 0:
        # Return capacity-weighted distance average
        return np.sum(distances * capacities) / sum_capacity
    else:
        # Fallback: Return unweighted average distance
        return np.mean(distances)


# ---- RUNNING THE PIPELINE ----

# Apply the distance calculation safely
final_df['capacity_weighted_distance'] = final_df.apply(compute_weighted_distance, axis=1)

# Fill Nans
final_df['global_event_count'] = final_df['global_event_count'].fillna(0).astype(int)
final_df['total_daily_capacity'] = final_df['total_daily_capacity'].fillna(0)

# Drop payload structure safely (errors='ignore' ensures it won't crash if already dropped)
final_df.drop(columns=['geo_payload'], errors='ignore', inplace=True)

final_df.head(3)


,listing_id,date,is_booked,day_of_week,id,neighborhood,price,latitude,longitude,event_date,global_event_count,total_daily_capacity,capacity_weighted_distance
0,6606,2026-08-01,0,Saturday,6606,Wallingford,111.77,47.65444,-122.33629,2026-08-01,1,68740.0,4.106972
1,6606,2026-08-02,0,Sunday,6606,Wallingford,111.77,47.65444,-122.33629,2026-08-02,1,17151.0,2.380312
2,6606,2026-08-03,0,Monday,6606,Wallingford,111.77,47.65444,-122.33629,2026-08-03,1,17151.0,2.380312


In [27]:
# Remove unnecessary columns
final_df = final_df.drop(columns=['latitude', 'longitude', 'event_date', 'id'])

final_df.info()
final_df.head()


<class 'pandas.DataFrame'>
RangeIndex: 215047 entries, 0 to 215046
Data columns (total 9 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   listing_id                  215047 non-null  int64         
 1   date                        215047 non-null  datetime64[us]
 2   is_booked                   215047 non-null  int64         
 3   day_of_week                 215047 non-null  str           
 4   neighborhood                215047 non-null  str           
 5   price                       215047 non-null  float64       
 6   global_event_count          215047 non-null  int64         
 7   total_daily_capacity        215047 non-null  float64       
 8   capacity_weighted_distance  124866 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(3), str(2)
memory usage: 14.8 MB


,listing_id,date,is_booked,day_of_week,neighborhood,price,global_event_count,total_daily_capacity,capacity_weighted_distance
0,6606,2026-08-01,0,Saturday,Wallingford,111.77,1,68740.0,4.106972
1,6606,2026-08-02,0,Sunday,Wallingford,111.77,1,17151.0,2.380312
2,6606,2026-08-03,0,Monday,Wallingford,111.77,1,17151.0,2.380312
3,6606,2026-08-04,0,Tuesday,Wallingford,111.77,1,17151.0,2.380312
4,6606,2026-08-05,0,Wednesday,Wallingford,111.77,1,17151.0,2.380312


In [28]:
# Save final_df to csv file
final_df.to_csv('seattle_events_airbnb.csv', index=False)